In [ ]:
import math
import torch
import torch.nn as nn

In [ ]:
class dot_product_attention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, mask=None):
        dim = q.size(-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(dim) # Compute scaled q @ kᵀ

        if mask != None:
            scores = scores.masked_fill(mask==0, float("-inf")) # Apply mask i.e preserve auto-regressive property

        scores = torch.softmax(scores, dim=-1)
        return torch.matmul(scores, v) # Compute weighted sum

class multi_head_attention(nn.Module):
    def __init__(self, n_heads=8, d_model=512):
        super().__init__()
        assert d_model % n_heads == 0 # Ensure each head gets an equal slice of the total dimension
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model) # Introduce linear projections for query, key, and value
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.attention = dot_product_attention()
        self.out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        q = self.w_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2) # Split into multiple heads then reorder for parallel attention
        k = self.w_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        x = self.attention(q, k, v, mask) # Applies attention for each head
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k) # Transpose back to initial shape
        return self.out(x)

class feed_forward_networks(nn.Module):
    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()
        self.out = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.out(x)

class positional_encoding(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        pe = torch.zeros(10000, d_model) # We assume a maximal sequence length of 10000
        position = torch.arange(0, 10000).unsqueeze(1) # Position indices from 0 to 9999
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)) # Scale the positions to ensure each dimension uses a different frequency
        pe[:, 0::2] = torch.sin(position * div_term) # Applies sin to even indices and cos to odd indices of the pe matrix
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe) # Register pe as a non-learnable parameter

    def forward(self, x):
        return x + self.pe[:, :x.size(1)] # Add the positional encoding to the input x

In [ ]:
class encoder_layer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048):
        super().__init__()
        self.attention = multi_head_attention(n_heads, d_model)
        self.out = feed_forward_networks(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, x, x, mask)) # Residual connection before each layer normalization
        return self.norm2(x + self.out(x))

class decoder_layer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048):
        super().__init__()
        self.attention = multi_head_attention(n_heads, d_model)
        self.cross_attention = multi_head_attention(n_heads, d_model)
        self.out = feed_forward_networks(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, encoder_mask=None, mask=None):
        x = self.norm1(x + self.attention(x, x, x, mask))
        x = self.norm2(x + self.cross_attention(x, encoder_output, encoder_output, encoder_mask)) # k, v from encoder and q from decoder
        return self.norm3(x + self.out(x))

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, N=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_enc = positional_encoding(d_model)
        self.layers = nn.ModuleList(
            [encoder_layer(d_model, n_heads, d_ff) for _ in range(N)]
        )

    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.positional_enc(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, N=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_enc = positional_encoding(d_model)
        self.layers = nn.ModuleList(
            [decoder_layer(d_model, n_heads, d_ff) for _ in range(N)]
        )

    def forward(self, x, encoder_output, encoder_mask=None, mask=None):
        x = self.embedding(x)
        x = self.positional_enc(x)
        for layer in self.layers:
            x = layer(x, encoder_output, encoder_mask, mask)
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, n_heads=8, d_ff=2048, N=6):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, N)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, N)
        self.out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, encoder_mask=None):
        encoder_output = self.encoder(src, src_mask)
        decoder_output = self.decoder(tgt, encoder_output, encoder_mask, tgt_mask)
        return self.out(decoder_output)

In [ ]:
model = Transformer(src_vocab_size=10000, tgt_vocab_size=10000)
src = torch.randint(0, 10000, (32, 20)) # Dummy dataset
tgt = torch.randint(0, 10000, (32, 20))
out = model(src, tgt)